In [3]:
import requests
import pandas as pd
import time

## Club

In [4]:
def extract_team(season):
    # Professional headers to mimic a browser
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Origin": "https://www.premierleague.com",
        "Referer": "https://www.premierleague.com/"
    }

    url = f'https://sdp-prem-prod.premier-league-prod.pulselive.com/api/v1/competitions/8/seasons/{season}/teams?_limit=20'
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()  # Check for errors
        data = response.json()
  
        teams_list = []
        
        # Iterate through the 'data' key in the JSON
        for team in data.get('data', []):
            stadium = team.get('stadium', {})
            
            record = {
                "team_id": team.get("id"),
                "season_id": season,
                "team_name": team.get("name"),
                "short_name": team.get("shortName"),
                "abbreviation": team.get("abbr"),
                "stadium_name": stadium.get("name"),
                "city": stadium.get("city"),
                "capacity": stadium.get("capacity"),
                "country": stadium.get("country")
            }
            teams_list.append(record)
        
        return pd.DataFrame(teams_list)
    except Exception as e:
        print(f"Failed to fetch club: {e}")

# Generate the DataFrame
df_club = extract_team(season=2015)

# Show result
df_club

,team_id,season_id,team_name,short_name,abbreviation,stadium_name,city,capacity,country
0,3,2015,Arsenal,Arsenal,ARS,Emirates Stadium,London,60272,England
1,7,2015,Aston Villa,A Villa,AVL,Villa Park,Birmingham,42682,England
2,91,2015,Bournemouth,Bournemouth,BOU,Vitality Stadium,Bournemouth,10783,England
3,8,2015,Chelsea,Chelsea,CHE,Stamford Bridge,London,41798,England
4,31,2015,Crystal Palace,C Palace,CRY,Selhurst Park,London,25073,England
5,11,2015,Everton,Everton,EVE,Goodison Park,Liverpool,39571,England
6,13,2015,Leicester City,Leicester,LEI,King Power Stadium,Leicester,32312,England
7,14,2015,Liverpool,Liverpool,LIV,Anfield,Liverpool,45276,England
8,43,2015,Manchester City,Man City,MCI,Etihad Stadium,Manchester,46708,England
9,1,2015,Manchester United,Man Utd,MUN,Old Trafford,Manchester,75635,England


In [5]:
for club in df_club['team_id'].unique()

array(['3', '7', '91', '8', '31', '11', '13', '14', '43', '1', '4', '45',
       '20', '110', '56', '80', '6', '57', '35', '21'], dtype=object)

## Player

In [48]:
def extract_squad(season_id, team_id):
    # Professional headers to mimic a browser
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Origin": "https://www.premierleague.com",
        "Referer": "https://www.premierleague.com/"
    }

    url = f'https://sdp-prem-prod.premier-league-prod.pulselive.com/api/v2/competitions/8/seasons/{season_id}/teams/{team_id}/squad'
    
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        data = response.json()
  
        players_list = []
        team_info = data.get('team', {})
        
        # Iterate through the 'players' key in the JSON
        for p in data.get('players', []):
            name_data = p.get('name', {})
            date_data = p.get('dates', {})
            country_data = p.get('country', {})
            
            record = {
                "team_id": team_id,
                "season_id": season_id,
                "team_name": team_info.get("name"),
                "player_id": p.get("id"),
                "display_name": name_data.get("display"),
                "first_name": name_data.get("first"),
                "last_name": name_data.get("last"),
                "position": p.get("position"),
                "shirt_num": p.get("shirtNum"),
                "birth_date": date_data.get("birth"),
                "joined_club": date_data.get("joinedClub"),
                "nationality": country_data.get("country"),
                "height_cm": p.get("height"),
                "weight_kg": p.get("weight"),
                "preferred_foot": p.get("preferredFoot"),
                "is_loan": p.get("loan")
            }
            players_list.append(record)
        
        return pd.DataFrame(players_list)
    
    except Exception as e:
        print(f"Failed to fetch squad for team {team_id}: {e}")
        return pd.DataFrame()

# Generate the DataFrame for Arsenal (ID: 3) in the 2025 season
df_squad = extract_squad(season_id=2025, team_id=3)

# Show result
df_squad.head()

,team_id,season_id,team_name,player_id,display_name,first_name,last_name,position,shirt_num,birth_date,joined_club,nationality,height_cm,weight_kg,preferred_foot,is_loan
0,3,2025,Arsenal,154561,David Raya,David,Raya,Goalkeeper,1,1995-09-15,2024-07-04,Spain,183.0,80.0,Right,0
1,3,2025,Arsenal,109745,Kepa Arrizabalaga,Kepa,Arrizabalaga,Goalkeeper,13,1994-10-03,2025-07-01,Spain,189.0,84.0,Right,0
2,3,2025,Arsenal,551221,Tommy Setford,Tommy,Setford,Goalkeeper,35,2006-03-13,2024-07-24,England,185.0,76.0,Right,0
3,3,2025,Arsenal,553747,Alexei Rojas,Alexei,Rojas,Goalkeeper,51,2005-09-28,2024-07-24,Colombia,187.0,77.0,Right,0
4,3,2025,Arsenal,616059,Jack Porter,Jack,Porter,Goalkeeper,78,2008-07-15,2024-09-18,England,185.0,75.0,Right,0


In [47]:
df_squad[df_squad['is_loan']==1]

,team_id,team_name,player_id,display_name,first_name,last_name,position,shirt_num,birth_date,joined_club,nationality,height_cm,weight_kg,preferred_foot,is_loan
7,3,Arsenal,500040,Cristhian Mosquera,Cristhian,Mosquera,Defender,3,2004-06-27,2025-07-24,Spain,191.0,78.0,Right,1
9,3,Arsenal,448104,Piero Hincapié,Piero,Hincapié,Defender,5,2002-01-09,2025-09-01,Ecuador,184.0,77.0,Left,1


## Match

In [50]:
def fetch_full_season_data(competition_id=8, season=2025):
    all_matches = []
    
    # Iterate through all 38 matchweeks
    for mw in range(1, 2):
        print(f"Fetching Matchweek {mw}...")
        
        url = f"https://sdp-prem-prod.premier-league-prod.pulselive.com/api/v2/matches"
        params = {
            "competition": competition_id,
            "season": season,
            "matchweek": mw,
            "_limit": 20
        }
        
        try:
            # Adding headers to prevent being blocked
            headers = {"User-Agent": "Mozilla/5.0"}
            response = requests.get(url, params=params, headers=headers)
            response.raise_for_status()
            data = response.json()
            
            matches = data.get('data', [])
            
            for m in matches:
                # Logic to handle missing scores for PreMatch/Postponed games
                home = m.get('homeTeam', {})
                away = m.get('awayTeam', {})
                
                match_entry = {
                    "match_id": m.get("matchId"),
                    "season_id": season,
                    "matchweek": m.get("matchWeek"),
                    "kickoff": m.get("kickoff"),
                    "status": m.get("period"), # e.g., FullTime, PreMatch
                    "result_type": m.get("resultType"), # e.g., NormalResult, Postponed
                    "ground": m.get("ground"),
                    
                    # Home Team Details
                    "home_team": home.get("name"),
                    "home_team_id": home.get("id"),
                    "home_abbr": home.get("abbr"),
                    "home_score_ht": home.get("halfTimeScore"),
                    "home_score": home.get("score") if m.get("period") == "FullTime" else None,
                    "home_redcard": home.get("redCards"),
                    
                    # Away Team Details
                    "away_team": away.get("name"),
                    "away_team_id": away.get("id"),
                    "away_abbr": away.get("abbr"),
                    "away_score_ht": away.get("halfTimeScore"),
                    "away_score": away.get("score") if m.get("period") == "FullTime" else None,
                    "away_redcard": away.get("redCards"),
                    
                    "attendance": m.get("attendance")
                }
                all_matches.append(match_entry)
            
            # Small sleep to be polite to the server
            time.sleep(0.5)
            
        except Exception as e:
            print(f"Failed to fetch matchweek {mw}: {e}")

    return pd.DataFrame(all_matches)

# Execute and Save
df_season = fetch_full_season_data(season=2010)

# Data Analyst Quality Check
# print(df_season.info())
# df_season.to_csv("pl_season_2025_26.csv", index=False)
df_season.head(20)

Fetching Matchweek 1...


,match_id,season_id,matchweek,kickoff,status,result_type,ground,home_team,home_team_id,home_abbr,home_score_ht,home_score,home_redcard,away_team,away_team_id,away_abbr,away_score_ht,away_score,away_redcard,attendance
0,321657,2010,1,2010-08-14 15:00:00,FullTime,NormalResult,"Villa Park, Birmingham",Aston Villa,7,AVL,2,3,0,West Ham United,21,WHU,0,0,0,36604
1,321658,2010,1,2010-08-14 15:00:00,FullTime,NormalResult,"Ewood Park, Blackburn",Blackburn Rovers,5,BLA,1,1,0,Everton,11,EVE,0,0,0,25869
2,321659,2010,1,2010-08-14 15:00:00,FullTime,NormalResult,"DW Stadium, Wigan",Wigan Athletic,111,WIG,0,0,0,Blackpool,92,BPL,3,4,0,16152
3,321660,2010,1,2010-08-14 15:00:00,FullTime,NormalResult,"Macron Stadium, Bolton",Bolton Wanderers,30,BOL,0,0,0,Fulham,54,FUL,0,0,0,20352
4,321661,2010,1,2010-08-14 17:30:00,FullTime,NormalResult,"Stamford Bridge, London",Chelsea,8,CHE,2,6,0,West Bromwich Albion,35,WBA,0,0,0,41589
5,321662,2010,1,2010-08-15 16:00:00,FullTime,NormalResult,"Anfield, Liverpool",Liverpool,14,LIV,0,1,1,Arsenal,3,ARS,0,1,1,44722
6,321663,2010,1,2010-08-16 20:00:00,FullTime,NormalResult,"Old Trafford, Manchester",Manchester United,1,MUN,2,3,0,Newcastle United,4,NEW,0,0,0,75221
7,321664,2010,1,2010-08-14 15:00:00,FullTime,NormalResult,"Stadium of Light, Sunderland",Sunderland,56,SUN,1,2,1,Birmingham City,41,BIR,0,2,0,38390
8,321665,2010,1,2010-08-14 12:45:00,FullTime,NormalResult,"White Hart Lane, London",Tottenham Hotspur,6,TOT,0,0,0,Manchester City,43,MCI,0,0,0,35928
9,321666,2010,1,2010-08-14 15:00:00,FullTime,NormalResult,"Molineux Stadium, Wolverhampton",Wolverhampton Wanderers,39,WOL,2,2,0,Stoke City,110,STO,0,1,0,27850


In [23]:
import requests
import pandas as pd
import time

class PremierLeagueScraper:
    def __init__(self, season):
        self.season = season
        self.base_api = "https://sdp-prem-prod.premier-league-prod.pulselive.com/api"
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            "Origin": "https://www.premierleague.com",
            "Referer": "https://www.premierleague.com/"
        }

    def get_teams(self):
        """Extracts all clubs participating in the specified season."""
        url = f"{self.base_api}/v1/competitions/8/seasons/{self.season}/teams?_limit=20"
        try:
            response = requests.get(url, headers=self.headers)
            response.raise_for_status()
            data = response.json()
            
            teams = []
            # for t in data:
            #     stadium = t.get('stadium', {})
            #     teams.append({
            #         "team_id": t.get("id"),
            #         "team_name": t.get("name"),
            #         "abbreviation": t.get("abbr"),
            #         "stadium": stadium.get("name"),
            #         "city": stadium.get("city"),
            #         "capacity": stadium.get("capacity")
            #     })
            for t in data.get('data', []):
                stadium = t.get('stadium', {})
                
                record = {
                    "team_id": t.get("id"),
                    "season_id": self.season,
                    "team_name": t.get("name"),
                    "short_name": t.get("shortName"),
                    "abbreviation": t.get("abbr"),
                    "stadium_name": stadium.get("name"),
                    "city": stadium.get("city"),
                    "capacity": stadium.get("capacity"),
                    "country": stadium.get("country")
                }
                teams.append(record)
            return pd.DataFrame(teams)
        except Exception as e:
            print(f"Error fetching teams: {e}")
            return pd.DataFrame()

    def get_all_squads(self, team_ids):
        """Loops through team IDs to get all players for the season."""
        all_players = []
        for t_id in team_ids:
            print(f"Fetching Squad for Team ID: {t_id}...")
            url = f"{self.base_api}/v2/competitions/8/seasons/{self.season}/teams/{t_id}/squad"
            try:
                res = requests.get(url, headers=self.headers)
                data = res.json()
                # for p in data.get('players', []):
                #     all_players.append({
                #         "team_id": t_id,
                #         "player_id": p.get("id"),
                #         "name": p.get("name", {}).get("display"),
                #         "position": p.get("position"),
                #         "height": p.get("height"),
                #         "weight": p.get("weight"),
                #         "nationality": p.get("country", {}).get("country")
                #     })
                team_info = data.get('team', {})

                for p in data.get('players', []):
                    name_data = p.get('name', {})
                    date_data = p.get('dates', {})
                    country_data = p.get('country', {})
                    
                    record = {
                        "team_id": t_id,
                        "season_id": self.season,
                        "team_name": team_info.get("name"),
                        "player_id": p.get("id"),
                        "display_name": name_data.get("display"),
                        "first_name": name_data.get("first"),
                        "last_name": name_data.get("last"),
                        "position": p.get("position"),
                        "shirt_num": p.get("shirtNum"),
                        "birth_date": date_data.get("birth"),
                        "joined_club": date_data.get("joinedClub"),
                        "nationality": country_data.get("country"),
                        "height_cm": p.get("height"),
                        "weight_kg": p.get("weight"),
                        "preferred_foot": p.get("preferredFoot"),
                        "is_loan": p.get("loan")
                    }
                    all_players.append(record)
                time.sleep(1) # Be polite
            except: continue
        return pd.DataFrame(all_players)

    def get_matches(self, matchweeks=1):
        """Fetches match results for the entire season."""
        all_matches = []
        url = f"{self.base_api}/v2/matches"
        
        for mw in range(1, matchweeks + 1):
            print(f"Fetching Matchweek {mw}...")
            params = {"competition": 8, "season": self.season, "matchweek": mw, "_limit": 20}
            try:
                res = requests.get(url, params=params, headers=self.headers)
                matches = res.json().get('data', [])
                for m in matches:
                    home, away = m.get('homeTeam', {}), m.get('awayTeam', {})
                    all_matches.append({
                    "match_id": m.get("matchId"),
                    "season_id": self.season,
                    "matchweek": m.get("matchWeek"),
                    "kickoff": m.get("kickoff"),
                    "status": m.get("period"), # e.g., FullTime, PreMatch
                    "result_type": m.get("resultType"), # e.g., NormalResult, Postponed
                    "ground": m.get("ground"),
                    
                    # Home Team Details
                    "home_team": home.get("name"),
                    "home_team_id": home.get("id"),
                    "home_abbr": home.get("abbr"),
                    "home_score_ht": home.get("halfTimeScore"),
                    "home_score": home.get("score") if m.get("period") == "FullTime" else None,
                    "home_redcard": home.get("redCards"),
                    
                    # Away Team Details
                    "away_team": away.get("name"),
                    "away_team_id": away.get("id"),
                    "away_abbr": away.get("abbr"),
                    "away_score_ht": away.get("halfTimeScore"),
                    "away_score": away.get("score") if m.get("period") == "FullTime" else None,
                    "away_redcard": away.get("redCards"),
                    
                    "attendance": m.get("attendance")
                    })
                time.sleep(1)
            except: continue
        return pd.DataFrame(all_matches)

In [24]:

# --- EXECUTION ---
scraper = PremierLeagueScraper(season=2025)

# 1. Get Clubs
df_clubs = scraper.get_teams()

# 2. Get All Squads based on the Clubs found
df_all_players = scraper.get_all_squads(df_clubs['team_id'].tolist())

# 3. Get Full Season Matches
df_season_matches = scraper.get_matches()

# # Now you have 3 clean DataFrames synced to 2025
# print(df_all_players.head())

Fetching Squad for Team ID: 3...
Fetching Squad for Team ID: 7...
Fetching Squad for Team ID: 91...
Fetching Squad for Team ID: 94...
Fetching Squad for Team ID: 36...
Fetching Squad for Team ID: 90...
Fetching Squad for Team ID: 8...
Fetching Squad for Team ID: 31...
Fetching Squad for Team ID: 11...
Fetching Squad for Team ID: 54...
Fetching Squad for Team ID: 2...
Fetching Squad for Team ID: 14...
Fetching Squad for Team ID: 43...
Fetching Squad for Team ID: 1...
Fetching Squad for Team ID: 4...
Fetching Squad for Team ID: 17...
Fetching Squad for Team ID: 56...
Fetching Squad for Team ID: 6...
Fetching Squad for Team ID: 21...
Fetching Squad for Team ID: 39...
Fetching Matchweek 1...


In [19]:
df_club

,team_id,season_id,team_name,short_name,abbreviation,stadium_name,city,capacity,country
0,3,2015,Arsenal,Arsenal,ARS,Emirates Stadium,London,60272,England
1,7,2015,Aston Villa,A Villa,AVL,Villa Park,Birmingham,42682,England
2,91,2015,Bournemouth,Bournemouth,BOU,Vitality Stadium,Bournemouth,10783,England
3,8,2015,Chelsea,Chelsea,CHE,Stamford Bridge,London,41798,England
4,31,2015,Crystal Palace,C Palace,CRY,Selhurst Park,London,25073,England
5,11,2015,Everton,Everton,EVE,Goodison Park,Liverpool,39571,England
6,13,2015,Leicester City,Leicester,LEI,King Power Stadium,Leicester,32312,England
7,14,2015,Liverpool,Liverpool,LIV,Anfield,Liverpool,45276,England
8,43,2015,Manchester City,Man City,MCI,Etihad Stadium,Manchester,46708,England
9,1,2015,Manchester United,Man Utd,MUN,Old Trafford,Manchester,75635,England


In [25]:
df_clubs

,team_id,season_id,team_name,short_name,abbreviation,stadium_name,city,capacity,country
0,3,2025,Arsenal,Arsenal,ARS,Emirates Stadium,London,60272,England
1,7,2025,Aston Villa,A Villa,AVL,Villa Park,Birmingham,42682,England
2,91,2025,Bournemouth,Bournemouth,BOU,Vitality Stadium,Bournemouth,11286,England
3,94,2025,Brentford,Brentford,BRE,Gtech Community Stadium,Brentford,17250,England
4,36,2025,Brighton and Hove Albion,Brighton,BHA,American Express Stadium,Falmer,30750,England
5,90,2025,Burnley,Burnley,BUR,Turf Moor,Burnley,21401,England
6,8,2025,Chelsea,Chelsea,CHE,Stamford Bridge,London,41798,England
7,31,2025,Crystal Palace,C Palace,CRY,Selhurst Park,London,25486,England
8,11,2025,Everton,Everton,EVE,Hill Dickinson Stadium,Liverpool,52888,England
9,54,2025,Fulham,Fulham,FUL,Craven Cottage,London,29130,England


In [26]:
df_season_matches

,match_id,season_id,matchweek,kickoff,status,result_type,ground,home_team,home_team_id,home_abbr,home_score_ht,home_score,home_redcard,away_team,away_team_id,away_abbr,away_score_ht,away_score,away_redcard,attendance
0,2561895,2025,1,2025-08-15 20:00:00,FullTime,NormalResult,"Anfield, Liverpool",Liverpool,14,LIV,1,4,0,Bournemouth,91,BOU,0,2,0,60315
1,2561896,2025,1,2025-08-16 12:30:00,FullTime,NormalResult,"Villa Park, Birmingham",Aston Villa,7,AVL,0,0,1,Newcastle United,4,NEW,0,0,0,42526
2,2561897,2025,1,2025-08-16 15:00:00,FullTime,NormalResult,"American Express Stadium, Falmer",Brighton and Hove Albion,36,BHA,0,1,0,Fulham,54,FUL,0,1,0,31478
3,2561898,2025,1,2025-08-17 14:00:00,FullTime,NormalResult,"The City Ground, Nottingham",Nottingham Forest,17,NFO,3,3,0,Brentford,94,BRE,0,1,0,29949
4,2561899,2025,1,2025-08-16 15:00:00,FullTime,NormalResult,"Stadium of Light, Sunderland",Sunderland,56,SUN,0,3,0,West Ham United,21,WHU,0,0,0,46233
5,2561900,2025,1,2025-08-16 15:00:00,FullTime,NormalResult,"Tottenham Hotspur Stadium, London",Tottenham Hotspur,6,TOT,1,3,0,Burnley,90,BUR,0,0,0,61077
6,2561901,2025,1,2025-08-16 17:30:00,FullTime,NormalResult,"Molineux Stadium, Wolverhampton",Wolverhampton Wanderers,39,WOL,0,0,0,Manchester City,43,MCI,2,4,0,31118
7,2561902,2025,1,2025-08-17 14:00:00,FullTime,NormalResult,"Stamford Bridge, London",Chelsea,8,CHE,0,0,0,Crystal Palace,31,CRY,0,0,0,39678
8,2561903,2025,1,2025-08-17 16:30:00,FullTime,NormalResult,"Old Trafford, Manchester",Manchester United,1,MUN,0,0,0,Arsenal,3,ARS,1,1,0,73475
9,2561904,2025,1,2025-08-18 20:00:00,FullTime,NormalResult,"Elland Road, Leeds",Leeds United,2,LEE,0,1,0,Everton,11,EVE,0,0,0,36820


In [27]:
df_all_players

,team_id,season_id,team_name,player_id,display_name,first_name,last_name,position,shirt_num,birth_date,joined_club,nationality,height_cm,weight_kg,preferred_foot,is_loan
0,3,2025,Arsenal,154561,David Raya,David,Raya,Goalkeeper,1.0,1995-09-15,2024-07-04,Spain,183.0,80.0,Right,0
1,3,2025,Arsenal,109745,Kepa Arrizabalaga,Kepa,Arrizabalaga,Goalkeeper,13.0,1994-10-03,2025-07-01,Spain,189.0,84.0,Right,0
2,3,2025,Arsenal,551221,Tommy Setford,Tommy,Setford,Goalkeeper,35.0,2006-03-13,2024-07-24,England,185.0,76.0,Right,0
3,3,2025,Arsenal,553747,Alexei Rojas,Alexei,Rojas,Goalkeeper,51.0,2005-09-28,2024-07-24,Colombia,187.0,77.0,Right,0
4,3,2025,Arsenal,616059,Jack Porter,Jack,Porter,Goalkeeper,78.0,2008-07-15,2024-09-18,England,185.0,75.0,Right,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
648,39,2025,Wolverhampton Wanderers,184754,Hwang Hee-Chan,Hee-Chan,Hwang,Forward,11.0,1996-01-26,2022-07-01,South Korea,177.0,77.0,Right,0
649,39,2025,Wolverhampton Wanderers,488213,Tolu Arokodare,Tolu,Arokodare,Forward,14.0,2000-11-23,2025-09-01,Nigeria,197.0,97.0,Right,0
650,39,2025,Wolverhampton Wanderers,593001,Enso González,Enso,González,Forward,30.0,2005-01-20,2023-08-31,Paraguay,169.0,69.0,Left,0
651,39,2025,Wolverhampton Wanderers,647671,Mateus Mané,Mateus,Mané,Forward,36.0,2007-09-16,2025-02-24,Portugal,175.0,NaN,Right,0
